# Robot leftright

In [1]:
pip install  mediapipe pymycobot pyserial


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import serial.tools.list_ports

# List all availablqe serial ports
ports = serial.tools.list_ports.comports()

for port, desc, hwid in sorted(ports):
    print(f"Port: {port}, Description: {desc}, Hardware ID: {hwid}")

Port: /dev/ttyACM0, Description: USB Single Serial, Hardware ID: USB VID:PID=1A86:55D4 SER=5901001678 LOCATION=3-8:1.0
Port: /dev/ttyS0, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS1, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS2, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS3, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS4, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS5, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS6, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS7, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS8, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS9, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS10, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS11, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS12, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS13, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS14, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS15, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS16, Descript

In [ ]:
import cv2
import mediapipe as mp
from google.protobuf.json_format import MessageToDict
from pymycobot import MyCobot
import serial
import time
import threading

# Initialize MediaPipe Hands with optimized settings
mpHands = mp.solutions.hands
hands = mpHands.Hands(
    static_image_mode=False,
    model_complexity=0,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7,
    max_num_hands=2
)

# Initialize the robot
try:
    mc = MyCobot('/dev/ttyACM0', 115200)
except serial.SerialException as e:
    print("Error:", e)
    exit()

# Start capturing video from webcam
cap = cv2.VideoCapture(2)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)   # Optional: Increase resolution if needed
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)            # Adjust FPS for smoother operation

current_hand = None  # Track current detected hand
lock = threading.Lock()  # Ensure thread safety for robot movements
last_command_time = 0  # Timestamp to debounce movement commands
COMMAND_DELAY = 0.5  # Minimum delay (in seconds) between commands

def move_robot(direction):
    """Move the robot based on hand direction."""
    with lock:  # Ensure only one thread sends commands at a time
        if direction == 'Right':
            mc.send_angles([90, 0, 90, -90, -90, 0], 40)
        elif direction == 'Left':
            mc.send_angles([90, 0, -90, 90, -90, 0], 40)
        elif direction == 'Origin':
            mc.send_angles([0, 0, 0, 0, 0, 0], 40)
        print(f"Robot moved: {direction}")
        time.sleep(0.1)  # Short delay to allow the robot to stabilize

def control_robot(direction):
    """Control the robot without blocking the main thread."""
    global last_command_time, current_hand
    current_time = time.time()
    
    if current_hand != direction and (current_time - last_command_time) > COMMAND_DELAY:
        current_hand = direction
        last_command_time = current_time
        threading.Thread(target=move_robot, args=(direction,)).start()

while True:
    success, img = cap.read()
    if not success:
        print("Failed to capture image")
        break

    img = cv2.flip(img, 1)
    img_resized = cv2.resize(img, (320, 240))  # Resize for efficiency

    imgRGB = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    results = hands.process(imgRGB)

    if results.multi_hand_landmarks:
        if len(results.multi_handedness) == 2:  # Both hands detected
            cv2.putText(img, 'Both Hands', (250, 50),
                        cv2.FONT_HERSHEY_COMPLEX, 0.9, (0, 255, 0), 2)
            control_robot('Origin')

        else:
            for i in results.multi_handedness:
                handedness_dict = MessageToDict(i)
                label = handedness_dict['classification'][0]['label']

                if label == 'Left':
                    cv2.putText(img, label + ' Hand', (20, 50),
                                cv2.FONT_HERSHEY_COMPLEX, 0.9, (0, 255, 0), 2)
                    control_robot('Left')

                elif label == 'Right':
                    cv2.putText(img, label + ' Hand', (460, 50),
                                cv2.FONT_HERSHEY_COMPLEX, 0.9, (0, 255, 0), 2)
                    control_robot('Right')

    cv2.imshow('Image', img)
    if cv2.waitKey(1) & 0xff == ord('q'):
        break

# Return to origin after ending the task
move_robot('Origin')

cap.release()
cv2.destroyAllWindows()

# Ensure the port is closed properly
if mc._serial_port.is_open:
    mc._serial_port.close()
    print("Serial port closed.")


2025-06-05 10:46:36.479671: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-05 10:46:36.480190: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-05 10:46:36.482727: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-05 10:46:36.489087: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749091596.499884    5558 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749091596.50

Note: This class is no longer maintained since v3.6.0, please refer to the project documentation: https://github.com/elephantrobotics/pymycobot/blob/main/README.md


W0000 00:00:1749091601.244398    5764 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Robot moved: Left
Robot moved: Origin
Robot moved: Left


# Use this for Demo

In [ ]:
import cv2
import mediapipe as mp
from google.protobuf.json_format import MessageToDict
from pymycobot import MyCobot
import serial
import time
import threading

# Initialize MediaPipe Hands with optimized settings
mpHands = mp.solutions.hands
hands = mpHands.Hands(
    static_image_mode=False,
    model_complexity=0,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7,
    max_num_hands=2
)

# Initialize the robot
try:
    mc = MyCobot('/dev/ttyACM0', 115200)
except serial.SerialException as e:
    print("Error:", e)
    exit()

# Start capturing video from webcam
cap = cv2.VideoCapture(2)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)   # Optional: Increase resolution if needed
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)            # Adjust FPS for smoother operation

current_hand = None  # Track current detected hand
last_hand_detected = None  # Track the last hand detected ('Right', 'Left', 'Both')
lock = threading.Lock()  # Ensure thread safety for robot movements
last_command_time = 0  # Timestamp to debounce movement commands
COMMAND_DELAY = 0.5  # Minimum delay (in seconds) between commands

def move_robot(direction):
    """Move the robot based on hand direction."""
    with lock:  # Ensure only one thread sends commands at a time
        if direction == 'Right':
            mc.send_angles([90, 0, 90, -90, -90, 0], 40)
        elif direction == 'Left':
            mc.send_angles([90, 0, -90, 90, -90, 0], 40)
        elif direction == 'Both':
            mc.send_angles([0, 0, 0, 0, 0, 0], 40)
        print(f"Robot moved: {direction}")
        time.sleep(0.1)  # Short delay to allow the robot to stabilize

def control_robot(direction):
    """Control the robot without blocking the main thread."""
    global last_command_time, last_hand_detected
    current_time = time.time()
    
    # Only move to a new position if the hand is different and enough time has passed
    if direction != last_hand_detected and (current_time - last_command_time) > COMMAND_DELAY:
        last_hand_detected = direction
        last_command_time = current_time
        threading.Thread(target=move_robot, args=(direction,)).start()

while True:
    success, img = cap.read()
    if not success:
        print("Failed to capture image")
        break

    img = cv2.flip(img, 1)
    img_resized = cv2.resize(img, (320, 240))  # Resize for efficiency

    imgRGB = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    results = hands.process(imgRGB)

    if results.multi_hand_landmarks:
        if len(results.multi_handedness) == 2:  # Both hands detected
            cv2.putText(img, 'Both Hands', (250, 50),
                        cv2.FONT_HERSHEY_COMPLEX, 0.9, (0, 255, 0), 2)
            if last_hand_detected != 'Both':  # Only go to "Both" if it was not already "Both"
                control_robot('Both')
                last_hand_detected = 'Both'

        else:
            for i in results.multi_handedness:
                handedness_dict = MessageToDict(i)
                label = handedness_dict['classification'][0]['label']

                if label == 'Left':
                    cv2.putText(img, label + ' Hand', (20, 50),
                                cv2.FONT_HERSHEY_COMPLEX, 0.9, (0, 255, 0), 2)
                    if last_hand_detected != 'Left':  # Only go to Left if it was not already Left
                        control_robot('Left')
                        last_hand_detected = 'Left'

                elif label == 'Right':
                    cv2.putText(img, label + ' Hand', (460, 50),
                                cv2.FONT_HERSHEY_COMPLEX, 0.9, (0, 255, 0), 2)
                    if last_hand_detected != 'Right':  # Only go to Right if it was not already Right
                        control_robot('Right')
                        last_hand_detected = 'Right'

    cv2.imshow('Image', img)
    if cv2.waitKey(1) & 0xff == ord('q'):
        break

# Return to origin after ending the task
move_robot('Both')

cap.release()
cv2.destroyAllWindows()

# Ensure the port is closed properly
if mc._serial_port.is_open:
    mc._serial_port.close()
    print("Serial port closed.")


Note: This class is no longer maintained since v3.6.0, please refer to the project documentation: https://github.com/elephantrobotics/pymycobot/blob/main/README.md


I0000 00:00:1747303378.554286    4880 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1747303378.559247   14179 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 24.2.8-1ubuntu1~24.04.1), renderer: Mesa Intel(R) Arc(tm) Graphics (MTL)
W0000 00:00:1747303378.575147   14155 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1747303378.582748   14165 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [ ]:
# import serial
from pymycobot import MyCobot
import time

try:
    
    mc = MyCobot('/dev/ttyACM0', 115200)
    mc.send_angles([0, 0, 0, 0, 0, 0], 40)

except serial.SerialException as e:
    print("Error:", e)

finally:
    # Ensure the port is closed properly
    if mc._serial_port.is_open:
        mc._serial_port.close()
        print("Serial port closed.")